### Basic Thread

In [4]:
import threading
import time

def worker(name, seconds):
    print(f"staring: {name}")
    time.sleep(seconds)
    print(f"done: {name}")

t1 = threading.Thread(target=worker, args=('T1', 2))
t2 = threading.Thread(target=worker, args=('T2', 1))

t1.start()
t2.start()

t1.join()
t2.join()

staring: T1
staring: T2
done: T2
done: T1


In [13]:
t = threading.Thread(target=worker, args=('Test', 300))
t.start()

staring: Test


In [18]:
t.is_alive()

True

### Race Condition

In [10]:
import threading
import time
import sys

# Force GIL to switch threads frequently
# sys.setswitchinterval(0.00001)

c = 0

def counter():
    global c

    for _ in range(10000):
        temp = c
        time.sleep(0)
        c = temp + 1

t1 = threading.Thread(target=counter)
t2 = threading.Thread(target=counter)

t1.start()
t2.start()

t1.join()
t2.join()

print(c)


##############
# Safe Counter
##############

c = 0
lock = threading.Lock()

def safeCounter():
    global c

    for _ in range(10000):
        with lock: # Acquire lock first and then increment
            temp = c
            time.sleep(0)
            c = temp + 1

t1 = threading.Thread(target=safeCounter)
t2 = threading.Thread(target=safeCounter)

t1.start()
t2.start()

t1.join()
t2.join()

print(c)

18246
20000


In [ ]:
lock = threading.Lock()

def work():
    i = 0
    for _ in range(10):
        print(i)
        i += 1
        time.sleep(10)

if lock.acquire(timeout = 2.0):
    try:
        work()
    finally:
        lock.release()
else:
    print("Timeout")

### Blocking Queue

In [28]:
import threading
from queue import Queue, Empty

bq: Queue[str] = Queue(maxsize=2)
stop = threading.Event()

def consumer():
    while not stop.is_set():
        try:
            item = bq.get(timeout=0.2)
        except Empty:
            print("queue is empty, waiting...")
            continue
        print(item)
        bq.task_done()

t = threading.Thread(target = consumer, daemon = True)
t.start()

queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
q

In [ ]:
for msg in ['A', 'B', 'C']:
    bq.put(msg)

A
B
C


queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...
queue is empty, waiting...


In [30]:
stop.set()
t.join()

queue is empty, waiting...


In [31]:
from queue import Full

bq.put("X")
bq.put("Y")

In [32]:
bq.qsize()

2

In [33]:
try:
    bq.put("Z", timeout=1)   # blocks 2s then raises Full
except Full:
    print(f"Queue Full")

Queue Full
